In [8]:
from warnings import filterwarnings
filterwarnings('ignore')

In [1]:
!uv pip install wikipedia
!uv pip install markitdown
!uv pip install rich

Using Python 3.12.3 environment at: /Users/shudhanshu/Desktop/Study Projects/GenAI/.venv
Resolved 9 packages in 567ms                                         
Prepared 1 package in 114ms                                              
Installed 1 package in 0.94ms                               
 + wikipedia==1.4.0
Using Python 3.12.3 environment at: /Users/shudhanshu/Desktop/Study Projects/GenAI/.venv
Resolved 22 packages in 206ms                                        
Prepared 3 packages in 2.45s                                             
Installed 3 packages in 12ms                                
 + magika==0.6.3
 + markdownify==1.2.2
 + markitdown==0.1.5
Using Python 3.12.3 environment at: /Users/shudhanshu/Desktop/Study Projects/GenAI/.venv
Checked 1 package in 8ms


In [39]:
import os
from dotenv import load_dotenv
load_dotenv()

True

In [3]:
from langchain_community.tools import WikipediaQueryRun
from langchain_community.utilities import WikipediaAPIWrapper


In [4]:
wiki_api_wrapper = WikipediaAPIWrapper(top_k_results=3, doc_content_chars_max=8000)
wiki_tool = WikipediaQueryRun(api_wrapper=wiki_api_wrapper, feature='lxml')

In [5]:
wiki_tool.description

'A wrapper around Wikipedia. Useful for when you need to answer general questions about people, places, companies, facts, historical events, or other subjects. Input should be a search query.'

In [6]:
wiki_tool.args

{'query': {'description': 'query to look up on wikipedia',
  'title': 'Query',
  'type': 'string'}}

In [9]:
print(wiki_tool.invoke({'query':'apple'}))

Page: Apple
Summary: An apple is the round, edible fruit of an apple tree (Malus spp.). Fruit trees of the orchard or domestic apple (Malus domestica), the most widely grown in the genus, are cultivated worldwide. The tree originated in Central Asia, where its wild ancestor, Malus sieversii, is still found. Apples have been grown for thousands of years in Eurasia before they were introduced to North America by European colonists. Apples have cultural significance in many mythologies (including Norse and Greek) and religions (such as Christianity in Europe).
Apples grown from seeds tend to be very different from those of their parents, and the resultant fruit frequently lacks desired characteristics. For commercial purposes, including botanical evaluation, apple cultivars are propagated by clonal grafting onto rootstocks. Apple trees grown without rootstocks tend to be larger and much slower to fruit after planting. Rootstocks are used to control the speed of growth and the size of the 

In [11]:
# Agent implemetation using langchain
# step 1 : Tool declaration

from langchain_core.tools import Tool
wiki_tool_init = Tool(
    name="Wikipedia",
    func=wiki_api_wrapper.run,
    description="useful when you need a detailed answer about general knowledge"
)

In [12]:
wiki_tool_init.description

'useful when you need a detailed answer about general knowledge'

In [13]:
wiki_tool_init.args

{'tool_input': {'type': 'string'}}

In [14]:
print(wiki_tool_init.invoke({'tool_input':"AI"}))

Page: Artificial intelligence
Summary: Artificial intelligence (AI) is the capability of computational systems to perform tasks typically associated with human intelligence, such as learning, reasoning, problem-solving, perception, and decision-making. It is a field of research in engineering, mathematics and computer science that develops and studies methods and software that enable machines to perceive their environment and use learning and intelligence to take actions that maximize their chances of achieving defined goals.
High-profile applications of AI include advanced web search engines, chatbots, virtual assistants, autonomous vehicles, and play and analysis in strategy games (e.g., chess and Go). Since the 2020s, generative AI has become widely available to generate images, audio, and videos from text prompts.
The traditional goals of AI research include learning, reasoning, knowledge representation, planning, natural language processing, and perception, as well as support for 

In [15]:
from langchain_community.tools.tavily_search import TavilySearchResults

In [16]:
tavily_tool = TavilySearchResults(
    max_results=8,
    search_depth='advance',
    include_raw_content=True
)

In [17]:
tavily_tool.args

{'query': {'description': 'search query to look up',
  'title': 'Query',
  'type': 'string'}}

In [18]:
tavily_tool.description

'A search engine optimized for comprehensive, accurate, and trusted results. Useful for when you need to answer questions about current events. Input should be a search query.'

In [19]:
result = tavily_tool.invoke('tell me about google')

In [20]:
result

"HTTPError('400 Client Error: Bad Request for url: https://api.tavily.com/search')"

In [22]:
# create own tool

from langchain_core.tools import tool

@tool
def multiply(a, b):
    """Multiply two numbers"""
    return a*b


print(multiply.name)
print(multiply.args)
print(multiply.description)

multiply
{'a': {'title': 'A'}, 'b': {'title': 'B'}}
Multiply two numbers


In [23]:
type(multiply)

langchain_core.tools.structured.StructuredTool

In [24]:
multiply.invoke({"a":5,"b":10})

50

In [26]:
# Tool with data type enforcing

from pydantic import BaseModel, Field
from langchain_core.tools import StructuredTool

class CalculatorInput(BaseModel):
    a:float = Field(description="first number")
    b:float = Field(description="second number")

def multiply(a:float,b:float)->float:
    """Multiply two floating numbers"""
    return a*b

multiply = StructuredTool.from_function(
    func=multiply,
    name="multiply",
    description="used to multiply two floating numbers",
    args_schema=CalculatorInput,
    return_direct=True
)




In [27]:
print(multiply.name)
print(multiply.args)
print(multiply.description)

multiply
{'a': {'description': 'first number', 'title': 'A', 'type': 'number'}, 'b': {'description': 'second number', 'title': 'B', 'type': 'number'}}
used to multiply two floating numbers


In [29]:
multiply.invoke({"a":10,"b":9})

90.0

In [44]:
# weather Tool

import requests

@tool
def get_weather(query:str)->list:
    """Search weatherapi to get the current weather."""
    base_url = "https://api.openweathermap.org/data/2.5/weather"
    complete_url = f"{base_url}?q={query}&APPID={os.environ.get('WEATHER_API_KEY')}"

    response = requests.get(complete_url)
    data = response.json()

    print(complete_url)
    print(data)
    if data.get('name'):
        return data
    else:
        return "weather data not found"
        

In [46]:
get_weather.invoke('Bangalore')

https://api.openweathermap.org/data/2.5/weather?q=Bangalore&APPID=8ce32efc60500f625bb6c5983ce35bee
{'coord': {'lon': 77.6033, 'lat': 12.9762}, 'weather': [{'id': 800, 'main': 'Clear', 'description': 'clear sky', 'icon': '01n'}], 'base': 'stations', 'main': {'temp': 301.99, 'feels_like': 301.43, 'temp_min': 301.49, 'temp_max': 302.77, 'pressure': 1010, 'humidity': 38, 'sea_level': 1010, 'grnd_level': 914}, 'visibility': 6000, 'wind': {'speed': 10.28, 'deg': 125, 'gust': 25.48}, 'clouds': {'all': 2}, 'dt': 1776183048, 'sys': {'type': 2, 'id': 2105374, 'country': 'IN', 'sunrise': 1776127058, 'sunset': 1776171711}, 'timezone': 19800, 'id': 1277333, 'name': 'Bengaluru', 'cod': 200}


{'coord': {'lon': 77.6033, 'lat': 12.9762},
 'weather': [{'id': 800,
   'main': 'Clear',
   'description': 'clear sky',
   'icon': '01n'}],
 'base': 'stations',
 'main': {'temp': 301.99,
  'feels_like': 301.43,
  'temp_min': 301.49,
  'temp_max': 302.77,
  'pressure': 1010,
  'humidity': 38,
  'sea_level': 1010,
  'grnd_level': 914},
 'visibility': 6000,
 'wind': {'speed': 10.28, 'deg': 125, 'gust': 25.48},
 'clouds': {'all': 2},
 'dt': 1776183048,
 'sys': {'type': 2,
  'id': 2105374,
  'country': 'IN',
  'sunrise': 1776127058,
  'sunset': 1776171711},
 'timezone': 19800,
 'id': 1277333,
 'name': 'Bengaluru',
 'cod': 200}

In [47]:
import rich

result = get_weather.invoke("delhi")

rich.print_json(data=result)

https://api.openweathermap.org/data/2.5/weather?q=delhi&APPID=8ce32efc60500f625bb6c5983ce35bee
{'coord': {'lon': 77.2167, 'lat': 28.6667}, 'weather': [{'id': 721, 'main': 'Haze', 'description': 'haze', 'icon': '50n'}], 'base': 'stations', 'main': {'temp': 304.2, 'feels_like': 302.68, 'temp_min': 304.2, 'temp_max': 304.2, 'pressure': 1006, 'humidity': 27, 'sea_level': 1006, 'grnd_level': 981}, 'visibility': 4000, 'wind': {'speed': 1.03, 'deg': 0}, 'clouds': {'all': 20}, 'dt': 1776183460, 'sys': {'type': 1, 'id': 9165, 'country': 'IN', 'sunrise': 1776126413, 'sunset': 1776172541}, 'timezone': 19800, 'id': 1273294, 'name': 'Delhi', 'cod': 200}


{
  "coord": {
    "lon": 77.2167,
    "lat": 28.6667
  },
  "weather": [
    {
      "id": 721,
      "main": "Haze",
      "description": "haze",
      "icon": "50n"
    }
  ],
  "base": "stations",
  "main": {
    "temp": 304.2,
    "feels_like": 302.68,
    "temp_min": 304.2,
    "temp_max": 304.2,
    "pressure": 1006,
    "humidity": 27,
    "sea_level": 1006,
    "grnd_level": 981
  },
  "visibility": 4000,
  "wind": {
    "speed": 1.03,
    "deg": 0
  },
  "clouds": {
    "all": 20
  },
  "dt": 1776183460,
  "sys": {
    "type": 1,
    "id": 9165,
    "country": "IN",
    "sunrise": 1776126413,
    "sunset": 1776172541
  },
  "timezone": 19800,
  "id": 1273294,
  "name": "Delhi",
  "cod": 200
}

In [49]:
from langchain_openai import ChatOpenAI
llm_gpt = ChatOpenAI(model='gpt-4o', temperature=0)

In [51]:
tools = [multiply, get_weather]

chatgpt_with_tools = llm_gpt.bind_tools(tools) #binding tool to the model

In [52]:
prompt = """
    Given only the tools at your disposal, mention tool calls for the following tasks:
    Do not change the query given for any search tasks
    1. what is 2.1 times 10
    2. what is the current weather in varanasi today
    3. what is the current weather in delhi today
    4. what is 2 times 5
"""

result = chatgpt_with_tools.invoke(prompt)

In [53]:
result

AIMessage(content='', additional_kwargs={'refusal': None}, response_metadata={'token_usage': {'completion_tokens': 83, 'prompt_tokens': 162, 'total_tokens': 245, 'completion_tokens_details': {'accepted_prediction_tokens': 0, 'audio_tokens': 0, 'reasoning_tokens': 0, 'rejected_prediction_tokens': 0}, 'prompt_tokens_details': {'audio_tokens': 0, 'cached_tokens': 0}}, 'model_provider': 'openai', 'model_name': 'gpt-4o-2024-08-06', 'system_fingerprint': 'fp_77c39d8bcb', 'id': 'chatcmpl-DUaqtTwz4hXBcUF8QsSq9l0wJS97C', 'service_tier': 'default', 'finish_reason': 'tool_calls', 'logprobs': None}, id='lc_run--019d8cce-fbc0-7981-a459-722a955a9e77-0', tool_calls=[{'name': 'multiply', 'args': {'a': 2.1, 'b': 10}, 'id': 'call_R2tA3SLuXXxYegMCmWewSFeW', 'type': 'tool_call'}, {'name': 'get_weather', 'args': {'query': 'varanasi'}, 'id': 'call_WbI5C7Y9IrPo8as9juvIfxF5', 'type': 'tool_call'}, {'name': 'get_weather', 'args': {'query': 'delhi'}, 'id': 'call_ralDK4cj0Za1vckMkANqtgkQ', 'type': 'tool_call'}, 

In [54]:
result.tool_calls

[{'name': 'multiply',
  'args': {'a': 2.1, 'b': 10},
  'id': 'call_R2tA3SLuXXxYegMCmWewSFeW',
  'type': 'tool_call'},
 {'name': 'get_weather',
  'args': {'query': 'varanasi'},
  'id': 'call_WbI5C7Y9IrPo8as9juvIfxF5',
  'type': 'tool_call'},
 {'name': 'get_weather',
  'args': {'query': 'delhi'},
  'id': 'call_ralDK4cj0Za1vckMkANqtgkQ',
  'type': 'tool_call'},
 {'name': 'multiply',
  'args': {'a': 2, 'b': 5},
  'id': 'call_S37PVr8WmyKRMqswzeMeUzbU',
  'type': 'tool_call'}]

In [55]:
toolkit = {
    "multiply":multiply,
    "get_weather":get_weather
}

for tool_call in result.tool_calls:
    selected_tool = toolkit[tool_call['name'].lower()]
    print(f"Calling Tool: {tool_call['name']}")

    tool_output = selected_tool.invoke(tool_call['args'])
    print(tool_output)

    print()

Calling Tool: multiply
21.0

Calling Tool: get_weather
https://api.openweathermap.org/data/2.5/weather?q=varanasi&APPID=8ce32efc60500f625bb6c5983ce35bee
{'coord': {'lon': 83, 'lat': 25.3333}, 'weather': [{'id': 721, 'main': 'Haze', 'description': 'haze', 'icon': '50n'}], 'base': 'stations', 'main': {'temp': 303.2, 'feels_like': 301.81, 'temp_min': 303.2, 'temp_max': 303.2, 'pressure': 1006, 'humidity': 28, 'sea_level': 1006, 'grnd_level': 997}, 'visibility': 3500, 'wind': {'speed': 1.54, 'deg': 310}, 'clouds': {'all': 0}, 'dt': 1776184075, 'sys': {'type': 1, 'id': 9138, 'country': 'IN', 'sunrise': 1776125198, 'sunset': 1776170980}, 'timezone': 19800, 'id': 1253405, 'name': 'Varanasi', 'cod': 200}
{'coord': {'lon': 83, 'lat': 25.3333}, 'weather': [{'id': 721, 'main': 'Haze', 'description': 'haze', 'icon': '50n'}], 'base': 'stations', 'main': {'temp': 303.2, 'feels_like': 301.81, 'temp_min': 303.2, 'temp_max': 303.2, 'pressure': 1006, 'humidity': 28, 'sea_level': 1006, 'grnd_level': 997}

In [56]:
# tool calling without native support for tool or function callling

In [59]:
from langchain_core.output_parsers import JsonOutputParser
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.tools import render_text_description

In [60]:
rendered_tools = render_text_description(tools)
print(rendered_tools)

multiply(a: float, b: float) -> float - used to multiply two floating numbers
get_weather(query: str) -> list - Search weatherapi to get the current weather.


In [61]:
system_prompt = f"""
    You are an assistant that has access to the following set of tools.
    Here are the names and description for each tool.

    {rendered_tools}

    Given the user instructions, for each instruction do the following:
        - Return the name and input of the tool to use.
        - Return your response as a JSON blob with 'name' and 'arguments' keys.
        - The 'arguments' should be a dictionary, with keys corresponding to the
        argument names and the values corresponding to the requested values
"""

prompt = ChatPromptTemplate.from_messages(
    [
        ("system", system_prompt),
        ("user","{input}")
    ]
)

In [62]:
instructions = [
    {"input":"what is 2.1 times 10"},
    {"input":"what is the current weather in varanasi today"},
    {"input":"what is the current weather in delhi today"},
    {"input":"what is 2 times 5"}
]

In [63]:
llm_chain = (
    prompt
    |
    llm_gpt
    |
    JsonOutputParser()
)


In [64]:
response = llm_chain.map().invoke(instructions)

In [65]:
response

[{'name': 'multiply', 'arguments': {'a': 2.1, 'b': 10}},
 {'name': 'get_weather', 'arguments': {'query': 'Varanasi'}},
 {'name': 'get_weather', 'arguments': {'query': 'Delhi'}},
 {'name': 'multiply', 'arguments': {'a': 2, 'b': 5}}]

In [67]:
toolkit = {
    "multiply":multiply,
    "get_weather":get_weather
}

for tool_call in response:
    selected_tool = toolkit[tool_call['name'].lower()]
    print(f"Calling Tool: {tool_call['name']}")

    tool_output = selected_tool.invoke(tool_call['arguments'])
    print(tool_output)

    print()

Calling Tool: multiply
21.0

Calling Tool: get_weather
https://api.openweathermap.org/data/2.5/weather?q=Varanasi&APPID=8ce32efc60500f625bb6c5983ce35bee
{'coord': {'lon': 83, 'lat': 25.3333}, 'weather': [{'id': 721, 'main': 'Haze', 'description': 'haze', 'icon': '50n'}], 'base': 'stations', 'main': {'temp': 301.2, 'feels_like': 300.49, 'temp_min': 301.2, 'temp_max': 301.2, 'pressure': 1005, 'humidity': 34, 'sea_level': 1005, 'grnd_level': 997}, 'visibility': 3500, 'wind': {'speed': 0, 'deg': 0}, 'clouds': {'all': 0}, 'dt': 1776184898, 'sys': {'type': 1, 'id': 9138, 'country': 'IN', 'sunrise': 1776125198, 'sunset': 1776170980}, 'timezone': 19800, 'id': 1253405, 'name': 'Varanasi', 'cod': 200}
{'coord': {'lon': 83, 'lat': 25.3333}, 'weather': [{'id': 721, 'main': 'Haze', 'description': 'haze', 'icon': '50n'}], 'base': 'stations', 'main': {'temp': 301.2, 'feels_like': 300.49, 'temp_min': 301.2, 'temp_max': 301.2, 'pressure': 1005, 'humidity': 34, 'sea_level': 1005, 'grnd_level': 997}, 'vi